# MCP + Agents AI Integration in Gemini
Author: arielzin33@gmail.com

**Theme: Workspace Assistant** — a Gemini-driven agent that composes **three MCP servers**:

| Server | Type | Tools | Purpose |
|---|---|---|---|
| `filesystem` | third-party (npx) | 14 | read/write/search files in the workspace |
| `git` | third-party (python) | 12 | status, diff, log, commit, branches |
| `custom_ops` | **custom (FastMCP)** | 4 | `ping`, `summarize_lines`, `word_frequency`, `summarize_file` |

The agent uses a **tool-driven policy** — Gemini decides which servers/tools to call and in what
order. Nothing is hard-coded into a fixed flow.

---
### Verified before publishing (run locally against these exact package versions)

All 30 tools were discovered and invoked for real; the composed client and Gemini agent were
built end-to-end. Three genuine breakages were found and fixed along the way — each documented
in the cell where it matters:

1. **FastMCP's startup banner corrupts stdio transport** — the decorative banner prints to
   *stdout*, the same channel the JSON-RPC protocol uses. The server starts fine, but the client
   dies with an opaque `McpError: Connection closed`. Fix: `mcp.run(transport="stdio", show_banner=False)`.
2. **`command: "python"` silently launches the wrong interpreter** — MCP spawns servers as
   subprocesses, and a bare `"python"` resolves against `PATH`. In a venv (or anywhere the active
   interpreter isn't first on `PATH`) that starts a *different* Python without `fastmcp`
   installed. Same opaque `Connection closed` error. Fix: `sys.executable`.
3. **`langchain-mcp-adapters==0.2.1` (the pinned version) is stale** — upgraded to `0.3.2`, which
   works against the current `mcp` package.

Colab note: Colab runs as root with a single system Python, so `sys.executable` and `"python"`
resolve to the same thing there — but `sys.executable` is correct in both environments, so it's
used throughout.

---
## 1. Install dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters>=0.3.2" \
  "fastmcp>=2.0.0" \
  "mcp-server-git" \
  "nest_asyncio"

# NOTE: the exercise pins langchain-mcp-adapters==0.2.1, but that version fails against the
# current `mcp` package with "McpError: Connection closed". 0.3.2 was verified working.
# mcp-server-git is added here because the git MCP server is a separate pip package.


---
## 2. Set `GOOGLE_API_KEY`

Get a free key at [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey). In Colab, store it via the 🔑 **Secrets** panel (name it `GOOGLE_API_KEY`) — the cell below reads it from there, falling back to a prompt.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Loaded GOOGLE_API_KEY from Colab secrets.")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        import getpass
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")
    print("GOOGLE_API_KEY set.")


---
## 3. Confirm Node/NPM (needed for the filesystem MCP server)

In [ ]:
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs npm)
!npx --version


---
## 4. Create the workspace

A real git repo with a couple of files, so the filesystem and git servers have something to actually operate on.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKDIR = Path("/content/workspace") if Path("/content").exists() else Path.cwd() / "workspace"
WORKDIR.mkdir(exist_ok=True)

(WORKDIR / "notes.txt").write_text(
    "project kickoff\n\nthe cat sat on the mat\nthe cat ran\n\nreview scheduled\n",
    encoding="utf-8",
)
(WORKDIR / "README.md").write_text("# Workspace\n\nDemo repo for the MCP agent.\n", encoding="utf-8")

subprocess.run(["git", "init", "-q"], cwd=WORKDIR)
subprocess.run(["git", "add", "."], cwd=WORKDIR)
subprocess.run(["git", "-c", "user.email=demo@example.com", "-c", "user.name=Demo",
                "commit", "-qm", "initial commit"], cwd=WORKDIR)

WORKDIR = str(WORKDIR)
print("Workspace ready:", WORKDIR)


---
## 5. Implement the custom MCP server (FastMCP)

Four tools relevant to the workspace-assistant theme. **`show_banner=False` is load-bearing** — see the note in the code.

In [ ]:
import textwrap
from pathlib import Path

server_path = Path(WORKDIR).parent / "custom_mcp_server.py"
server_path.write_text(textwrap.dedent('''
    from pathlib import Path
    from typing import Dict, List

    from fastmcp import FastMCP

    mcp = FastMCP(name="custom_ops")


    @mcp.tool
    def ping() -> str:
        """Health check tool."""
        return "pong"


    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        """Return counts about a list of lines: total and non-empty."""
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        return {"total_lines": total, "nonempty_lines": nonempty}


    @mcp.tool
    def word_frequency(text: str, top_n: int = 5) -> Dict[str, int]:
        """Return the top_n most frequent words in text."""
        freq: Dict[str, int] = {}
        for w in text.lower().split():
            w = w.strip(".,!?;:\\"'()[]")
            if w:
                freq[w] = freq.get(w, 0) + 1
        return dict(sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:top_n])


    @mcp.tool
    def summarize_file(path: str, max_lines: int = 20) -> Dict[str, object]:
        """Read a text file and return line count, word count, and a short preview."""
        p = Path(path)
        if not p.exists():
            return {"error": f"File not found: {path}"}
        text = p.read_text(encoding="utf-8", errors="replace")
        lines = text.splitlines()
        return {
            "line_count": len(lines),
            "word_count": len(text.split()),
            "preview": "\\n".join(lines[:max_lines]),
        }


    if __name__ == "__main__":
        # show_banner=False is REQUIRED for stdio transport. FastMCP's decorative startup
        # banner prints to stdout -- the exact channel the JSON-RPC protocol uses -- so
        # leaving it on corrupts the stream. The server appears to start fine, but the
        # client fails with an opaque "McpError: Connection closed". Confirmed directly.
        mcp.run(transport="stdio", show_banner=False)
'''), encoding="utf-8")

print("Wrote:", server_path)


---
## 6. Connect to all three MCP servers

`sys.executable` rather than a bare `"python"` — see the note in the code for why that distinction caused a hard-to-diagnose failure.

In [ ]:
import shutil, sys
from langchain_mcp_adapters.client import MultiServerMCPClient

NPX = shutil.which("npx") or "npx"

mcp_connections = {
    # --- third-party server #1: filesystem (Node, via npx) ---
    "filesystem": {
        "transport": "stdio",
        "command": NPX,
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    # --- third-party server #2: git (Python) ---
    "git": {
        "transport": "stdio",
        # sys.executable, NOT "python": MCP spawns this as a subprocess and a bare
        # "python" resolves against PATH, which in a venv can launch a different
        # interpreter that lacks the server package -- surfacing only as an opaque
        # "McpError: Connection closed" with no hint at the real cause.
        "command": sys.executable,
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    # --- our custom server #3 ---
    "custom_ops": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)


### Load the tools

`get_tools()` is async, and Colab may already have a running event loop. The helper below handles both cases.

**Why not just `nest_asyncio.apply()` + `asyncio.run(...)`?** That's the usual snippet, but calling `nest_asyncio.apply()` when *no* loop is running leaves `asyncio.run` in a state that fails with `anyio.NoEventLoopError` when the MCP client tries to spawn its subprocesses — hit this for real while testing the notebook outside Colab. Applying `nest_asyncio` only when a loop is actually running works in both environments.

In [ ]:
import asyncio


def run_async(coro):
    """Run a coroutine from a notebook cell or a plain script."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)          # plain script: no loop running yet
    import nest_asyncio                   # Colab/Jupyter: loop already running
    nest_asyncio.apply()
    return loop.run_until_complete(coro)


tools = run_async(client.get_tools())

print("Total tools:", len(tools))
for prefix in ("filesystem", "git", "custom_ops"):
    names = [t.name for t in tools if t.name.startswith(prefix)]
    print(f"  {prefix:12s} {len(names):2d} -> {names[:4]}{' ...' if len(names) > 4 else ''}")


**Real captured output:**
```
Total tools: 30
  filesystem   14 -> ['filesystem_read_file', 'filesystem_read_text_file', 'filesystem_read_media_file', 'filesystem_read_multiple_files'] ...
  git          12 -> ['git_git_status', 'git_git_diff_unstaged', 'git_git_diff_staged', 'git_git_diff'] ...
  custom_ops    4 -> ['custom_ops_ping', 'custom_ops_summarize_lines', 'custom_ops_word_frequency', 'custom_ops_summarize_file']
```

---
## 7. Build the Gemini agent

The system prompt describes *capabilities*, not a fixed procedure — Gemini decides which servers to call and in what order. That's the tool-driven policy the brief asks for.

In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

SYSTEM_PROMPT = (
    "You are a workspace assistant with access to three tool servers:\n"
    "- filesystem tools: read, write, list, and search files in the workspace\n"
    "- git tools: inspect status, diffs, log, and history of the repo\n"
    "- custom_ops tools: ping, summarize_lines, word_frequency, summarize_file\n\n"
    "Decide yourself which tools to call and in what order. Chain them when a task needs "
    "it -- for example, read a file with the filesystem tools, then pass its contents to "
    "custom_ops for analysis. Prefer real tool calls over guessing. State which tools you "
    "used, and if a task can't be done with the available tools, say so plainly."
)

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.2)
agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

print("Agent ready with", len(tools), "tools.")


---
## 8. Run the agent

Each task is designed to require a *different* combination of servers, so you can watch the routing policy actually make choices.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


def ask(task: str):
    print("=" * 78)
    print("TASK:", task)
    result = agent.invoke({"messages": [HumanMessage(content=task)]},
                          config={"recursion_limit": 25})
    msgs = result["messages"]
    used = [m.name for m in msgs if isinstance(m, ToolMessage)]
    if used:
        print("TOOLS CALLED:", used)
    final = msgs[-1]
    print("ANSWER:", final.content if isinstance(final, AIMessage) else final)
    print()


# single server (custom_ops)
ask("Health check: is the custom_ops server responding?")

# filesystem -> custom_ops composition
ask("Read notes.txt from the workspace and tell me its 3 most frequent words.")

# git server
ask("What's the current git status of the repo, and what was the last commit?")

# all three: filesystem write -> git status -> custom_ops analysis
ask("Create a file called summary.md containing a one-line description of this repo, "
    "then check whether git sees it as untracked, then report how many lines it has.")


---
## Troubleshooting

**`McpError: Connection closed`** — the catch-all failure for stdio MCP. It means the subprocess
died or wrote non-JSON to stdout. Three real causes hit while building this:

1. *FastMCP banner on stdout* — fix with `mcp.run(transport="stdio", show_banner=False)`.
2. *Wrong interpreter* — a bare `"python"` resolved to an interpreter without `fastmcp`. Use
   `sys.executable`.
3. *Stale adapter* — `langchain-mcp-adapters==0.2.1` fails against the current `mcp`; use `>=0.3.2`.

To debug, run the server directly (`!python custom_mcp_server.py`) and watch for anything printed
to stdout that isn't JSON-RPC.

**`npx` hangs or 404s** — the first `npx -y @modelcontextprotocol/server-filesystem` downloads the
package; allow it a minute. Loading all three servers took ~30s cold locally.

**Tool names look doubled (`git_git_status`)** — expected. `tool_name_prefix=True` prepends the
server name, and the git server's own tools are already named `git_*`.

---
## Notes on the design

- **Tool-driven, not hard-coded.** The system prompt lists capabilities and lets Gemini plan.
  Task 4 above requires chaining all three servers, and nothing in the code sequences those calls.
- **Custom tools chosen to compose.** `summarize_file` and `word_frequency` deliberately consume
  output the *filesystem* server produces, so cross-server chaining is possible rather than each
  server living in its own silo.
- **What still needs your key.** Tool discovery and invocation were verified for real (all 30
  tools). The agent was constructed and reached Gemini's API. What was *not* verified is Gemini's
  actual routing quality on these four tasks — that needs a valid `GOOGLE_API_KEY`.